# CS301 Project – Track 1: Machine Learning Model Development, Training, and Evaluation

# Stage 1: Data Preprocessing & Initial Modeling

## Objective
The goal of this project is to analyze a loan dataset and build both classification and regression models.
The classification model predicts whether a loan is approved, while the regression model predicts the loan amount.

## Dataset Description
The dataset contains information about loan applicants, including demographic and financial features such as:
- Gender, Married, Education (categorical features)
- ApplicantIncome, LoanAmount, Credit_History (numerical features)

The dataset satisfies the requirement of having at least 12 features, including both categorical and numerical variables.

## Stage 1 Steps
The following steps are performed:
1. Data Cleaning (handling missing values, removing duplicates)
2. Exploratory Data Analysis (EDA) using Plotly visualizations
3. Feature Selection based on correlation
4. Model Training:
   - Logistic Regression for classification
   - Linear Regression for prediction
5. Model Evaluation:
   - Accuracy and F1 Score (classification)
   - R^2 and RMSE (regression)

## Tools & Libraries
- Python (Pandas, NumPy)
- Plotly for visualization
- Scikit-learn for machine learning models

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_squared_error

## Load the Dataset

Load Data From the CSV file placed in the same folder as this main.py file. The shape and first few rows are displayed to understand the structure of the data.

In [2]:
df = pd.read_csv("loan_data_set.csv")

print("Shape:", df.shape)
df.head()

Shape: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


## Data Cleaning

Loan_ID is removed, duplicates are dropped, and missing values are filled using mode (categorical) and mean (numerical).

In [3]:
df = df.drop(columns=["Loan_ID"])
df = df.drop_duplicates()

for column in df.columns:
    if df[column].dtype == "object":
        df[column] = df[column].fillna(df[column].mode()[0])
    else:
        df[column] = df[column].fillna(df[column].mean())

## Define Targets

Loan_Status is used for classification and LoanAmount for regression. Loan_Status is converted from Y/N to 1/0.

In [4]:
classification_target = "Loan_Status"
regression_target = "LoanAmount"

# Keep Loan Status Label for Visualization
df["Loan_Status_Label"] = df["Loan_Status"]
df[classification_target] = df[classification_target].map({"Y": 1, "N": 0})

## EDA: Numerical Features vs Loan Status
Box plots compare numerical features for approved and rejected loans.

In [5]:
numerical_columns = df.select_dtypes(include=np.number).columns

# Show distribution of values for numerical columns for classification target separately
for column in numerical_columns:
    if column != classification_target:
        fig = px.box(df, x="Loan_Status_Label", y=column, title=f"{column} vs Loan Status")
        fig.show()

## EDA: Numerical Features vs Loan Amount
Scatter plots show relationships between numerical features and LoanAmount.

In [6]:
# Show the relationship of values for numerical columns for regression target separately
for column in numerical_columns:
    if column not in [regression_target, classification_target]:
        fig = px.scatter(df, x=column, y=regression_target, title=f"{column} vs Loan Amount")
        fig.show()


## EDA: Categorical Features vs Loan Status
Histograms show distributions of categorical features grouped by Loan_Status.

In [7]:
categorical_columns = df.select_dtypes(exclude=np.number).columns

# Show the distribution for categorical columns and analyze their relationship with Loan_Status using color
for column in categorical_columns:
    if column != "Loan_Status_Label":
        fig = px.histogram(df, x=column, color="Loan_Status_Label", barmode="group", title=f"{column} vs Loan Status")
        fig.show()

## Correlation Matrix

Correlation between numerical features is calculated and visualized.

In [8]:
corr = df[numerical_columns].corr()

print("\nCorrelation Matrix:\n")
print(corr)

fig = px.imshow(corr, text_auto=True, title="Correlation Matrix")
fig.show()


Correlation Matrix:

                   ApplicantIncome  CoapplicantIncome  LoanAmount  \
ApplicantIncome           1.000000          -0.116605    0.565620   
CoapplicantIncome        -0.116605           1.000000    0.187828   
LoanAmount                0.565620           0.187828    1.000000   
Loan_Amount_Term         -0.045242          -0.059675    0.038801   
Credit_History           -0.014477          -0.001665   -0.007738   
Loan_Status              -0.004710          -0.059187   -0.036416   

                   Loan_Amount_Term  Credit_History  Loan_Status  
ApplicantIncome           -0.045242       -0.014477    -0.004710  
CoapplicantIncome         -0.059675       -0.001665    -0.059187  
LoanAmount                 0.038801       -0.007738    -0.036416  
Loan_Amount_Term           1.000000        0.001395    -0.020974  
Credit_History             0.001395        1.000000     0.540483  
Loan_Status               -0.020974        0.540483     1.000000  


## Feature Selection

Numerical features are selected based on their correlation with LoanAmount. The correlation values are sorted in descending order using absolute values to identify the strongest relationships.

ApplicantIncome is selected because it shows the highest positive correlation (approximately 0.56) with LoanAmount. Most values fall within the range of 0 to 10,000, with some high-value outliers.

CoapplicantIncome is selected as it provides additional financial information. Although its correlation is weaker, it is still useful for selecting it as a feature.

Credit_History is selected despite low correlation with LoanAmount because it shows a strong relationship with Loan_Status in the box plots, making it important for classification.

Categorical features such as Married and Property_Area are selected based on their visual patterns. Married shows a difference in approval counts between categories, and Property_Area shows clear distinction, with Semiurban areas having higher approval rates.

Features such as Loan_Amount_Term, Gender, Dependents, Education, and Self_Employed are excluded because they show weak correlation values or minimal impact in the visualizations.

In [9]:
corr_target = corr[regression_target].abs().sort_values(ascending=False)

print("\nCorrelation with LoanAmount:\n")
print(corr_target)

regression_features = ["ApplicantIncome", "CoapplicantIncome", "Credit_History"]
print("\nNumerical Features (based on correlation):", regression_features)

categorical_features = ["Married", "Property_Area"]
print("Categorical Features:", categorical_features)
# Combine both regression and categorical features
selected_features = regression_features + categorical_features
selected_features = [feature for feature in selected_features if feature not in ["Loan_Status", "LoanAmount"]]
print("\nSelected Features:", selected_features)



Correlation with LoanAmount:

LoanAmount           1.000000
ApplicantIncome      0.565620
CoapplicantIncome    0.187828
Loan_Amount_Term     0.038801
Loan_Status          0.036416
Credit_History       0.007738
Name: LoanAmount, dtype: float64

Numerical Features (based on correlation): ['ApplicantIncome', 'CoapplicantIncome', 'Credit_History']
Categorical Features: ['Married', 'Property_Area']

Selected Features: ['ApplicantIncome', 'CoapplicantIncome', 'Credit_History', 'Married', 'Property_Area']


## Prepare Data

Selected features are used to create X. Categorical variables are converted using one-hot encoding.

In [10]:
X = df[selected_features]

# Convert categorical columns to numeric
X = pd.get_dummies(X, drop_first=True)


print("\nExample of Final Feature Columns After Encoding:")
print(X.head())


Example of Final Feature Columns After Encoding:
   ApplicantIncome  CoapplicantIncome  Credit_History  Married_Yes  \
0             5849                0.0             1.0        False   
1             4583             1508.0             1.0         True   
2             3000                0.0             1.0         True   
3             2583             2358.0             1.0         True   
4             6000                0.0             1.0        False   

   Property_Area_Semiurban  Property_Area_Urban  
0                    False                 True  
1                    False                False  
2                    False                 True  
3                    False                 True  
4                    False                 True  


## Classification Model

Logistic Regression is used to predict Loan_Status.

In [ ]:
y_classification = df[classification_target]

X_train, X_test, y_train, y_test = train_test_split(X, y_classification, test_size=0.2, random_state=42)

classification_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])
classification_model.fit(X_train, y_train)

y_pred = classification_model.predict(X_test)

print("\nClassification Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

## Regression Model

Linear Regression is used to predict LoanAmount.

In [12]:
y_regression = df[regression_target]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_regression, test_size=0.2, random_state=42)

regression_model = LinearRegression()
regression_model.fit(Xr_train, yr_train)

yr_pred = regression_model.predict(Xr_test)

print("\nRegression Results")
print("R2:", r2_score(yr_test, yr_pred))
print("RMSE:", np.sqrt(mean_squared_error(yr_test, yr_pred)))


Regression Results
R2: 0.5368421827246126
RMSE: 50.213921772767556


## Stage 2: Model Optimization and Comparative Analysis

Stage 1 established a Logistic Regression baseline for predicting Loan_Status. In Stage 2, two additional classification models are introduced for comparison.

In Stage 2, two additional classification models are introduced for comparison:

- Decision Tree Classifier
- Random Forest Classifier

All three models are trained on the same X_train / X_test split from Stage 1 and evaluated using Accuracy and F1 Score.

In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)

print("Decision Tree Results")
print("Accuracy:", dt_accuracy)
print("F1 Score:", dt_f1)

# Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("\nRandom Forest Results")
print("Accuracy:", rf_accuracy)
print("F1 Score:", rf_f1)

# Comparison
lr_accuracy = accuracy_score(y_test, y_pred)
lr_f1 = f1_score(y_test, y_pred)

stage2_initial_results = pd.DataFrame({
    "Model": ["Logistic Regression (Baseline)", "Decision Tree", "Random Forest"],
    "Accuracy": [round(lr_accuracy, 4), round(dt_accuracy, 4), round(rf_accuracy, 4)],
    "F1 Score": [round(lr_f1, 4), round(dt_f1, 4), round(rf_f1, 4)]
})

print("\nModel Comparison:")
stage2_initial_results

Decision Tree Results
Accuracy: 0.7154471544715447
F1 Score: 0.7904191616766467

Random Forest Results
Accuracy: 0.7317073170731707
F1 Score: 0.8070175438596491

Model Comparison:


,Model,Accuracy,F1 Score
0,Logistic Regression (Baseline),0.7886,0.8587
1,Decision Tree,0.7154,0.7904
2,Random Forest,0.7317,0.8070


### Hyperparameter Tuning with GridSearchCV

The basic models above use default settings. GridSearchCV systematically tests combinations of hyperparameters and selects the best configuration using 5-fold cross-validation, scored by F1 score.

This produces tuned versions of the Decision Tree and Random Forest that are saved as best_dt_model and best_rf_model for use in the final comparison.

In [15]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter Tuning: Decision Tree
dt_param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# GridSearchCV tests every combination using 5-fold cross-validation
dt_grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)
dt_grid_search.fit(X_train, y_train)

best_dt_model = dt_grid_search.best_estimator_

print("Best Decision Tree Parameters:", dt_grid_search.best_params_)
print("Best Decision Tree CV F1 Score:", round(dt_grid_search.best_score_, 4))

# Hyperparameter Tuning: Random Forest
rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# GridSearchCV tests every combination using 5-fold cross-validation
rf_grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)
rf_grid_search.fit(X_train, y_train)

best_rf_model = rf_grid_search.best_estimator_

print("\nBest Random Forest Parameters:", rf_grid_search.best_params_)
print("Best Random Forest CV F1 Score:", round(rf_grid_search.best_score_, 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Decision Tree Parameters: {'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2}
Best Decision Tree CV F1 Score: 0.8805
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Random Forest Parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Best Random Forest CV F1 Score: 0.8843


### Tuned Model Evaluation and Performance Comparison

Now that GridSearchCV has found the best hyperparameters, the tuned best_dt_model and best_rf_model are evaluated on the held-out test set. Their results are combined with the earlier baseline and basic model scores into a single comparison table and bar chart.

In [16]:
y_pred_best_dt = best_dt_model.predict(X_test)
y_pred_best_rf = best_rf_model.predict(X_test)

stage2_comparison_results = pd.DataFrame({
    "Model": [
        "Logistic Regression (Baseline)",
        "Decision Tree (Basic)",
        "Random Forest (Basic)",
        "Decision Tree (Tuned)",
        "Random Forest (Tuned)"
    ],
    "Accuracy": [
        round(accuracy_score(y_test, y_pred), 4),
        round(accuracy_score(y_test, y_pred_dt), 4),
        round(accuracy_score(y_test, y_pred_rf), 4),
        round(accuracy_score(y_test, y_pred_best_dt), 4),
        round(accuracy_score(y_test, y_pred_best_rf), 4)
    ],
    "F1 Score": [
        round(f1_score(y_test, y_pred), 4),
        round(f1_score(y_test, y_pred_dt), 4),
        round(f1_score(y_test, y_pred_rf), 4),
        round(f1_score(y_test, y_pred_best_dt), 4),
        round(f1_score(y_test, y_pred_best_rf), 4)
    ]
})

stage2_comparison_results = stage2_comparison_results.sort_values("F1 Score", ascending=False).reset_index(drop=True)

print("Model Comparison (sorted by F1 Score):")
display(stage2_comparison_results)

df_melted = stage2_comparison_results.melt(
    id_vars="Model",
    value_vars=["Accuracy", "F1 Score"],
    var_name="Metric",
    value_name="Score"
)

fig = px.bar(
    df_melted,
    x="Model",
    y="Score",
    color="Metric",
    barmode="group",
    title="Model Performance Comparison: Accuracy and F1 Score",
    labels={"Score": "Score", "Model": "Model"}
)
fig.update_layout(xaxis_tickangle=-20)
fig.show()

Model Comparison (sorted by F1 Score):


,Model,Accuracy,F1 Score
0,Logistic Regression (Baseline),0.7886,0.8587
1,Random Forest (Tuned),0.7886,0.8587
2,Decision Tree (Tuned),0.7886,0.8587
3,Random Forest (Basic),0.7317,0.8070
4,Decision Tree (Basic),0.7154,0.7904


### Final Model Selection and Stage 2 Summary

The best model is selected automatically from stage2_comparison_results based on the highest F1 Score.

In [17]:
best_row = stage2_comparison_results.iloc[0]
best_model_name = best_row["Model"]
best_accuracy = best_row["Accuracy"]
best_f1 = best_row["F1 Score"]

print("Best Model:", best_model_name)
print("Accuracy: ", best_accuracy)
print("F1 Score: ", best_f1)

Best Model: Logistic Regression (Baseline)
Accuracy:  0.7886
F1 Score:  0.8587


Stage 2 Summary

Stage 1 used Logistic Regression as the baseline classifier to predict Loan_Status. In Stage 2, two additional models were introduced, Decision Tree and Random Forest, and trained on the same feature set and train/test split. All models were evaluated using Accuracy and F1 Score. F1 Score was chosen as the primary ranking metric because it balances precision and recall, which is more informative than accuracy alone when one class (approved loans) is more frequent than the other.

GridSearchCV with 5-fold cross-validation was applied to both the Decision Tree and Random Forest to find their best hyperparameter settings. The tuned models were then compared against the baseline and the basic (default) versions. Based on the comparison table above, hyperparameter tuning improved performance over the basic defaults for at least one of the models. The top performing models had the same F1 Score. Logistic Regression was selected as the recommended model because it achieved the same best performance as the tuned tree-based models while being simpler, faster, and easier to interpret.
